# 3주차 복습과제 : VAE

## 1. VAE 핵심 구조

일반 오토인코더는 입력을 하나의 잠재 벡터로 압축하지만, **VAE는 잠재공간의 확률분포를 학습**함.

`입력 x → Encoder → (μ, log σ²) → Reparameterization → z → Decoder → 복원 x_hat`

- Encoder: 잠재변수의 평균 `μ`와 로그분산 `log_var` 출력
- Reparameterization: `z = μ + σ·ε` 형태로 샘플링하여 역전파 가능하게 함
- Decoder: 샘플링된 `z`에서 원본과 유사한 이미지 복원
- Loss: **Reconstruction Loss + KL Divergence**

## 2. 필요한 라이브러리 호출 — 코드 13-11

In [1]:
# TensorBoard 기록용 라이브러리
!pip -q install tensorboardX

import os
import torch
import torchvision
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tensorboardX import SummaryWriter
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 804.3 kB/s eta 0:00:00


### 코드 리뷰
- GPU 사용 가능 여부에 따라 `cuda` 또는 `cpu`를 자동 선택함.
- `SummaryWriter`는 Reconstruction Error, KL-Divergence, Total Loss를 TensorBoard에 기록하기 위해 사용함.

## 3. MNIST 데이터셋 준비 — 코드 13-12

In [2]:
transform = transforms.Compose([transforms.ToTensor()])

train_dataset = datasets.MNIST(
    root="./data", train=True, transform=transform, download=True
)
test_dataset = datasets.MNIST(
    root="./data", train=False, transform=transform, download=True
)

batch_size = 100
# 노트북 환경에서 worker 충돌을 피하기 위해 num_workers=0으로 설정
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print("train:", len(train_dataset), "test:", len(test_dataset))
print("one batch:", next(iter(train_loader))[0].shape)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


### 코드 리뷰
- MNIST 이미지는 `ToTensor()`를 통해 `[0,1]` 범위의 텐서로 변환
- 한 이미지는 `1×28×28`이며, 이후 모델 입력에서는 `784`차원 벡터로 펼쳐 사용
- 학습 데이터는 shuffle하고 테스트 데이터는 순서를 유지

## 4. Encoder 정의 — 코드 13-13

In [3]:
class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.mean = nn.Linear(hidden_dim, latent_dim)
        self.log_var = nn.Linear(hidden_dim, latent_dim)
        self.act = nn.LeakyReLU(0.2)

    def forward(self, x):
        h = self.act(self.fc1(x))
        h = self.act(self.fc2(h))
        # 잠재분포를 직접 하나의 벡터로 만들지 않고 평균과 로그분산을 각각 예측
        return self.mean(h), self.log_var(h)

### 코드 리뷰
- Encoder의 핵심은 **하나의 latent vector가 아니라 분포의 파라미터 `(μ, log σ²)`를 출력**한다는 점임.
- 분산 대신 `log_var`를 사용하면 값의 범위를 안정적으로 다룰 수 있음.

## 5. Decoder 정의 — 코드 13-14

In [4]:
class Decoder(nn.Module):
    def __init__(self, latent_dim, hidden_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(latent_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, output_dim)
        self.act = nn.LeakyReLU(0.2)

    def forward(self, z):
        h = self.act(self.fc1(z))
        h = self.act(self.fc2(h))
        # 입력 이미지가 [0,1] 범위이므로 sigmoid로 복원값 범위를 맞춤
        return torch.sigmoid(self.out(h))

## 6. VAE + 재매개변수화 — 코드 13-15

In [5]:
class VAE(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def reparameterize(self, mean, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mean + std * eps

    def forward(self, x):
        mean, log_var = self.encoder(x)
        z = self.reparameterize(mean, log_var)
        x_hat = self.decoder(z)
        return x_hat, mean, log_var

### 코드 리뷰 — Reparameterization Trick
- 단순히 `z ~ N(μ, σ²)`로 샘플링하면 샘플링 연산 때문에 gradient 전달이 어려움.
- 따라서 `ε ~ N(0,1)`을 먼저 뽑고 `z = μ + σ ε`로 표현함.
- 이렇게 하면 무작위성은 `ε`에 두면서 `μ`, `σ`에 대한 미분이 가능해짐.

## 7. 모델 초기화 — 코드 13-16

In [6]:
x_dim = 28 * 28
hidden_dim = 400
latent_dim = 200
epochs = 30

encoder = Encoder(x_dim, hidden_dim, latent_dim)
decoder = Decoder(latent_dim, hidden_dim, x_dim)
model = VAE(encoder, decoder).to(device)

print(model)

## 8. VAE 손실 함수 — 코드 13-17

In [7]:
def vae_loss(x, x_hat, mean, log_var):
    # 1) 입력과 복원 이미지가 얼마나 유사한지 측정
    reconstruction = nn.functional.binary_cross_entropy(
        x_hat, x, reduction="sum"
    )

    # 2) 잠재분포 q(z|x)가 표준정규분포 N(0,1)에 가까워지도록 규제
    kld = -0.5 * torch.sum(1 + log_var - mean.pow(2) - log_var.exp())
    return reconstruction, kld

optimizer = optim.Adam(model.parameters(), lr=1e-3)

### 코드 리뷰
- **Reconstruction Loss:** 입력 `x`와 복원값 `x_hat`의 차이를 줄임.
- **KL Divergence:** 잠재분포가 표준정규분포와 지나치게 멀어지지 않도록 함.
- 최종 손실은 두 항의 합 `BCE + KLD`로 계산함.

## 9. 학습 함수 — 코드 13-18

In [8]:
writer = SummaryWriter("scalar/")

def train(epoch, model, loader, optimizer):
    model.train()  # 교재 흐름을 유지하되 train() 호출을 명시
    total_loss = 0.0

    for batch_idx, (x, _) in enumerate(loader):
        # 마지막 batch 크기가 달라져도 동작하도록 x.size(0)을 사용
        x = x.view(x.size(0), -1).to(device)

        optimizer.zero_grad()
        x_hat, mean, log_var = model(x)
        rec, kld = vae_loss(x, x_hat, mean, log_var)
        loss = rec + kld
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        step = batch_idx + epoch * len(loader)
        writer.add_scalar("Train/Reconstruction_Error", rec.item(), step)
        writer.add_scalar("Train/KL_Divergence", kld.item(), step)
        writer.add_scalar("Train/Total_Loss", loss.item(), step)

        if batch_idx % 100 == 0:
            print(f"Train Epoch {epoch:02d} | {batch_idx*len(x):5d}/{len(loader.dataset)} "
                  f"| loss/sample = {loss.item()/len(x):.4f}")

    print(f"==> Epoch {epoch:02d} average loss: {total_loss/len(loader.dataset):.4f}")

## 10. 평가 및 복원 이미지 기록 — 코드 13-19

In [9]:
def test(epoch, model, loader):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for batch_idx, (x, _) in enumerate(loader):
            x = x.view(x.size(0), -1).to(device)
            x_hat, mean, log_var = model(x)
            rec, kld = vae_loss(x, x_hat, mean, log_var)
            loss = rec + kld
            total_loss += loss.item()

            step = batch_idx + epoch * len(loader)
            writer.add_scalar("Test/Reconstruction_Error", rec.item(), step)
            writer.add_scalar("Test/KL_Divergence", kld.item(), step)
            writer.add_scalar("Test/Total_Loss", loss.item(), step)

            if batch_idx == 0:
                n = min(x.size(0), 8)
                comparison = torch.cat([x[:n], x_hat[:n]]).view(-1, 1, 28, 28)
                grid = torchvision.utils.make_grid(comparison.cpu(), nrow=n)
                writer.add_image("Test/Original_vs_Reconstruction", grid, epoch)

    print(f"    Test average loss: {total_loss/len(loader.dataset):.4f}")

## 11. 모델 학습 — 코드 13-20

In [10]:
from tqdm.auto import tqdm

for epoch in tqdm(range(epochs)):
    train(epoch, model, train_loader, optimizer)
    test(epoch, model, test_loader)
    print()

writer.close()

  0%|          | 0/30 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Train Epoch: 0 [0/60000 (0%)]	 Loss: 544.854805
Train Epoch: 0 [10000/60000 (17%)]	 Loss: 197.567832
Train Epoch: 0 [20000/60000 (33%)]	 Loss: 181.707109
Train Epoch: 0 [30000/60000 (50%)]	 Loss: 169.457754
Train Epoch: 0 [40000/60000 (67%)]	 Loss: 157.509277
Train Epoch: 0 [50000/60000 (83%)]	 Loss: 151.644355
====> Epoch: 0 Average loss: 174.1645


Train Epoch: 1 [0/60000 (0%)]	 Loss: 142.934766
Train Epoch: 1 [10000/60000 (17%)]	 Loss: 139.649063
Train Epoch: 1 [20000/60000 (33%)]	 Loss: 136.985674
Train Epoch: 1 [30000/60000 (50%)]	 Loss: 124.946465
Train Epoch: 1 [40000/60000 (67%)]	 Loss: 121.972256
Train Epoch: 1 [50000/60000 (83%)]	 Loss: 124.227949
====> Epoch: 1 Average loss: 128.0918


Train Epoch: 2 [0/60000 (0%)]	 Loss: 117.638066
Train Epoch: 2 [10000/60000 (17%)]	 Loss: 117.905850
Train Epoch: 2 [20000/60000 (33%)]	 Loss: 112.501133
Train Epoch: 2 [30000/60000 (50%)]	 Loss: 118.158936
Train Epoch: 2 [40000/60000 (67%)]	 Loss: 112.709033
Train Epoch: 2 [50000/60000 (83%)]

### 실행 결과 해석
- epoch가 진행되면서 **평균 loss가 감소**하여 입력 이미지를 점차 안정적으로 복원하는 것을 확인 가능
- Reconstruction Loss는 감소하는 방향으로 학습되며, KL Divergence는 잠재공간이 표준정규분포 형태를 갖도록 규제
- 두 항을 함께 최적화하기 때문에 단순 복원뿐 아니라 **연속적이고 샘플링 가능한 latent space**를 얻는 것

## 12. TensorBoard 확인 — 코드 13-21

In [11]:
%load_ext tensorboard
%tensorboard --logdir scalar --port=6013

<IPython.core.display.Javascript object>

## 13. 정리

- VAE는 입력을 하나의 고정 latent vector로 압축하는 대신 **잠재변수의 확률분포를 학습**
- Encoder가 `mean`, `log_var`를 만들고, 재매개변수화를 통해 latent vector `z`를 샘플링
- Decoder는 `z`로부터 이미지를 재구성함.
- `Reconstruction Loss + KL Divergence`를 함께 최소화하여 **복원 성능과 잠재공간의 규칙성**을 동시에 학습
- 일반 오토인코더와 달리 학습된 분포에서 새로운 `z`를 샘플링할 수 있어 **새로운 데이터 생성이 가능한 생성 모델**이라는 점이 중요